# Lunar Road Demo: SysML v2 model

Run all cells in order. Each model cell should report the package it created; any error names the line.

## 01_Structure

In [ ]:
package LRD_Structure {
    doc /*
     * Lunar Road Demo MVP: logical architecture of the simulation system.
     * Source: "Lunar Road Demo MVP: System Boundary and Simulation Conditions".
     * Sourced values are bound with '='. Assumed values use 'default' so they
     * stay configurable, as the spec requires.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;

    // =====================================================================
    // External context: things the system uses but cannot change
    // =====================================================================

    part def LunarTerrain {
        doc /* NASA enhanced 5 m/pixel model for Connecting Ridge, LOLA Site01 as fallback. */
        port terrainOut : TerrainPort;
    }

    part def SunAndEarthEphemeris {
        doc /* SPICE kernels or a table giving Sun and Earth azimuth and elevation. */
        port ephemerisOut : EphemerisPort;
    }

    part def DemoViewer {
        doc /* Person running the comparison and changing the road assumptions. */
    }

    // =====================================================================
    // Items carried on the interfaces (IF-01 to IF-07)
    // =====================================================================

    item def TerrainData {
        doc /* IF-01: height (m), slope and wider horizon terrain. GeoTIFF on the LOLA polar stereographic grid.
             * The 5 m cell size applies to the 2 km patch; the wider horizon terrain is lower resolution. */
        attribute patchCellSize : LengthValue = 5 [m];
    }

    item def EphemerisData {
        doc /* IF-02: time, Sun and Earth azimuth and elevation (degrees). */
        attribute time : DurationValue;
        attribute sunAzimuth : AngularMeasureValue;
        attribute sunElevation : AngularMeasureValue;
        attribute earthAzimuth : AngularMeasureValue;
        attribute earthElevation : AngularMeasureValue;
    }

    item def EnvironmentMaps {
        doc /* IF-03: slope, lit flag, Earth flag and road flag by cell and time. Gridded arrays. */
        attribute slope : AngularMeasureValue;
        attribute isLit : Boolean;
        attribute earthInView : Boolean;
        attribute isRoad : Boolean;
    }

    item def Command {
        doc /* IF-04: planner to rover, through the delay. JSON message. */
        attribute commandId : String;
        attribute targetX : LengthValue;
        attribute targetY : LengthValue;
        attribute heading : AngularMeasureValue;
        attribute mode : String;
    }

    item def RoverStatus {
        doc /* IF-05: rover to planner, through the delay. JSON message. */
        attribute time : DurationValue;
        attribute x : LengthValue;
        attribute y : LengthValue;
        attribute heading : AngularMeasureValue;
        attribute stateOfCharge : Real;
        attribute mode : String;
        attribute commandStatus : String;
    }

    item def RunRecord {
        doc /* IF-06: time-stamped route, state, solar power, load, events and results. Parquet or CSV. */
        attribute time : DurationValue;
        attribute x : LengthValue;
        attribute y : LengthValue;
        attribute stateOfCharge : Real;
        attribute solarPower : PowerValue;
        attribute load : PowerValue;
        attribute eventText : String;
    }

    item def ScenarioSettings {
        doc /* IF-07: road on or off, preset, start, goal, start time, seed and parameters. Versioned YAML. */
        attribute roadOn : Boolean;
        attribute preset : String;
        attribute startX : LengthValue;
        attribute startY : LengthValue;
        attribute goalX : LengthValue;
        attribute goalY : LengthValue;
        attribute startTime : String;
        attribute seed : Integer;
    }

    // =====================================================================
    // Ports
    // =====================================================================

    port def TerrainPort      { out item terrain : TerrainData; }
    port def EphemerisPort    { out item ephemeris : EphemerisData; }
    port def EnvMapsPort      { out item maps : EnvironmentMaps; }
    port def CommandPort      { out item command : Command; }
    port def StatusPort       { out item status : RoverStatus; }
    port def RunRecordPort    { out item record : RunRecord; }
    port def ScenarioPort     { out item settings : ScenarioSettings; }

    // =====================================================================
    // Interface definitions, one per IF in the spec
    // =====================================================================

    interface def 'IF-01 TerrainInterface'   { end source : TerrainPort;   end sink : ~TerrainPort; }
    interface def 'IF-02 EphemerisInterface' { end source : EphemerisPort; end sink : ~EphemerisPort; }
    interface def 'IF-03 EnvMapsInterface'   { end source : EnvMapsPort;   end sink : ~EnvMapsPort; }
    interface def 'IF-04 CommandInterface'   { end source : CommandPort;   end sink : ~CommandPort; }
    interface def 'IF-05 StatusInterface'    { end source : StatusPort;    end sink : ~StatusPort; }
    interface def 'IF-06 RunRecordInterface' { end source : RunRecordPort; end sink : ~RunRecordPort; }
    interface def 'IF-07 ScenarioInterface'  { end source : ScenarioPort;  end sink : ~ScenarioPort; }

    // =====================================================================
    // Logical elements (the five simulation models plus display and config)
    // =====================================================================

    part def EnvironmentModel {
        doc /* Model 3: Sun and Earth visibility from terrain horizons; road cells from a fixed centreline. */
        attribute mapArea : LengthValue default 2000 [m];
        attribute roadWidth : LengthValue default 10 [m];
        attribute environmentStep : DurationValue default 15 [min];
        attribute earthVisibilityHeight : LengthValue default 2 [m];
        attribute sunlightHeight : LengthValue default 1 [m];
        port terrainIn : ~TerrainPort;
        port ephemerisIn : ~EphemerisPort;
        port mapsOut : EnvMapsPort;
        port settingsIn : ~ScenarioPort;
    }

    part def MobilityModel {
        doc /* Model 1: speed made good on flat ground, reduced by slip on slopes. */
        attribute nominalSpeed : SpeedValue = 10 [cm/s];
        attribute topSpeed : SpeedValue = 20 [cm/s];
        attribute offRoadSpeedMadeGood : SpeedValue = 0.8 [cm/s];
        attribute slopeLimit : AngularMeasureValue = 15 ['°'];
        attribute maxHeadingOffset : AngularMeasureValue default 50 ['°'];
        attribute offRoadRollingResistance : Real default 0.2;
        attribute movingFraction : Real;   // constant for each preset (REQ-P1)
        attribute currentSlope : AngularMeasureValue;
        attribute slip : Real;            // off-road: 2% at 0, 5% at 5, 10% at 10, 20% at 15 degrees (assumed)
    }

    part def PowerModel {
        doc /* Model 2: battery state updated each step as solar input minus load. */
        attribute batteryCapacity : EnergyValue = 5420 [W*h];   // at start of life, 0 C
        attribute solarPerPanel : PowerValue = 320 [W];
        attribute solarAtCorner : PowerValue = 450 [W];
        attribute batteryReserveFraction : Real default 0.2;
        attribute awakeLoad : PowerValue default 250 [W];
        attribute drivetrainEfficiency : Real default 0.5;
        attribute stateOfCharge : Real;
    }

    part def RoverModel {
        doc /* One VIPER-class rover. Mass and footprint from NASA 2024 slides (source A). */
        attribute rovingMass : MassValue = 447 [kg];
        attribute footprint : LengthValue = 1.7 [m];
        part mobility : MobilityModel;
        part power : PowerModel;
        port mapsIn : ~EnvMapsPort;
        port commandIn : ~CommandPort;
        port statusOut : StatusPort;
        port recordOut : RunRecordPort;
        port settingsIn : ~ScenarioPort;
    }

    part def CommandDelayLink {
        doc /* Model 4: each message waits half of a sampled 6 to 10 s round trip; optional drops. */
        attribute minRoundTrip : DurationValue = 6 [s];
        attribute maxRoundTrip : DurationValue = 10 [s];
        attribute statusPeriod : DurationValue default 60 [s];
        attribute ackTimeout : DurationValue default 10 [s];
        attribute packetDropRate : Real default 0.0;
        port mapsIn : ~EnvMapsPort;
        port commandIn : ~CommandPort;
        port commandOut : CommandPort;
        port statusIn : ~StatusPort;
        port statusOut : StatusPort;
        port settingsIn : ~ScenarioPort;
    }

    part def Planner {
        doc /* Model 5: the same time-aware planner runs in both cases, so any difference comes from the road. */
        attribute commandCycle : DurationValue = 5 [min];
        attribute minWaypointSpacing : LengthValue = 4 [m];   // off-road, sourced
        attribute maxWaypointSpacing : LengthValue = 8 [m];   // off-road, sourced; road limit set by preset
        attribute energyPenalty : Real default 0.001;   // hours per Wh drawn
        attribute maxPlanTime : DurationValue default 5 [min];
        attribute maxPlannedSlope : AngularMeasureValue;
        port mapsIn : ~EnvMapsPort;
        port commandOut : CommandPort;
        port statusIn : ~StatusPort;
        port recordOut : RunRecordPort;
        port settingsIn : ~ScenarioPort;
    }

    part def DisplayAndLog {
        doc /* Browser viewer over the stored run set, plus the run log. */
        attribute storedRuns : Integer = 24;
        port plannerRecordIn : ~RunRecordPort;
        port roverRecordIn : ~RunRecordPort;
        port settingsIn : ~ScenarioPort;
    }

    part def ScenarioConfig {
        doc /* Versioned YAML scenario: start, goal, start time, seed, road preset. */
        attribute scenarioLength : DurationValue default 96 [h];
        attribute startToGoal : LengthValue default 1500 [m];
        attribute initialCharge : Real default 1.0;
        port settingsOut : ScenarioPort;
    }

    // =====================================================================
    // The system of interest, with every interface connected
    // =====================================================================

    part def LunarRoadDemoSystem {
        doc /* Simulates one rover, its route and the assumed effects of a road. */
        part environment : EnvironmentModel;
        part planner : Planner;
        part commLink : CommandDelayLink;
        part rover : RoverModel;
        part display : DisplayAndLog;
        part scenario : ScenarioConfig;

        // Boundary ports: external inputs delegated to the environment model
        port terrainIn : ~TerrainPort;
        port ephemerisIn : ~EphemerisPort;
        bind terrainIn = environment.terrainIn;
        bind ephemerisIn = environment.ephemerisIn;

        // IF-03: environment to planner, link and rover
        interface envToPlanner : 'IF-03 EnvMapsInterface' connect
            source ::> environment.mapsOut to sink ::> planner.mapsIn;
        interface envToLink : 'IF-03 EnvMapsInterface' connect
            source ::> environment.mapsOut to sink ::> commLink.mapsIn;
        interface envToRover : 'IF-03 EnvMapsInterface' connect
            source ::> environment.mapsOut to sink ::> rover.mapsIn;

        // IF-04: planner to rover, through the delay
        interface cmdToLink : 'IF-04 CommandInterface' connect
            source ::> planner.commandOut to sink ::> commLink.commandIn;
        interface cmdToRover : 'IF-04 CommandInterface' connect
            source ::> commLink.commandOut to sink ::> rover.commandIn;

        // IF-05: rover to planner, through the delay
        interface statusToLink : 'IF-05 StatusInterface' connect
            source ::> rover.statusOut to sink ::> commLink.statusIn;
        interface statusToPlanner : 'IF-05 StatusInterface' connect
            source ::> commLink.statusOut to sink ::> planner.statusIn;

        // IF-06: planner and rover to display and log
        interface plannerToLog : 'IF-06 RunRecordInterface' connect
            source ::> planner.recordOut to sink ::> display.plannerRecordIn;
        interface roverToLog : 'IF-06 RunRecordInterface' connect
            source ::> rover.recordOut to sink ::> display.roverRecordIn;

        // IF-07: scenario settings to all blocks
        interface settingsToEnv : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> environment.settingsIn;
        interface settingsToPlanner : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> planner.settingsIn;
        interface settingsToRover : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> rover.settingsIn;
        interface settingsToLink : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> commLink.settingsIn;
        interface settingsToDisplay : 'IF-07 ScenarioInterface' connect
            source ::> scenario.settingsOut to sink ::> display.settingsIn;
    }

    // =====================================================================
    // Context: the system with its external inputs (IF-01, IF-02)
    // =====================================================================

    part demoContext {
        part terrain : LunarTerrain;
        part ephemeris : SunAndEarthEphemeris;
        part viewer : DemoViewer;
        part sim : LunarRoadDemoSystem;

        interface terrainLink : 'IF-01 TerrainInterface' connect
            source ::> terrain.terrainOut to sink ::> sim.terrainIn;
        interface ephemerisLink : 'IF-02 EphemerisInterface' connect
            source ::> ephemeris.ephemerisOut to sink ::> sim.ephemerisIn;
    }
}


## 02_Context_UseCases

In [ ]:
package LRD_Context {
    doc /*
     * Mission context and use cases for the Lunar Road Demo MVP.
     * The demo is complete when a viewer can run both cases, change the road
     * assumptions and see how the result changes.
     */

    public import LRD_Structure::*;

    use case def 'Compare Trip With And Without Road' {
        subject sim : LunarRoadDemoSystem;
        actor viewer : DemoViewer;
        actor terrain : LunarTerrain;
        actor ephemeris : SunAndEarthEphemeris;
        objective {
            doc /* Show the difference a road makes to the same VIPER-class trip in time,
                 * energy, risk and reliability, on real south polar terrain. */
        }
    }

    use case def 'Change Road Assumptions' {
        subject sim : LunarRoadDemoSystem;
        actor viewer : DemoViewer;
        objective {
            doc /* Switch road on or off, pick the half, expected or high road preset, and pick
                 * a 150 W, 250 W or 350 W awake load, then see results update from the stored run set. */
        }
    }

    use case def 'Inspect Run Results' {
        subject sim : LunarRoadDemoSystem;
        actor viewer : DemoViewer;
        objective {
            doc /* Read arrival time, distance, energy used, minimum state of charge, time in shadow,
                 * waiting time, peak slip and pass or fail for each run. */
        }
    }

    use case 'run the demo' : 'Compare Trip With And Without Road' {
        first start;
        then include use case 'change assumptions' : 'Change Road Assumptions';
        then include use case 'inspect results' : 'Inspect Run Results';
        then done;
    }
}


## 03_Requirements_Sim

In [ ]:
package LRD_Requirements_Sim {
    doc /*
     * Simulation requirements, taken word for word in meaning from the MVP spec.
     * REQ-M, REQ-P, REQ-E, REQ-C and REQ-R IDs match the spec. The spec gives no IDs
     * to its other "shall" statements, so REQ-MVP1 to REQ-MVP10 are assigned here.
     * Where a requirement has a number, it is also written as a checkable constraint.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;
    public import LRD_Structure::*;

    requirement def SimRequirement {
        doc /* Every simulation requirement is placed on the whole demo system. */
        subject sim : LunarRoadDemoSystem;
    }

    // ---------------------------------------------------------------------
    // Purpose and reporting
    // ---------------------------------------------------------------------

    requirement <'REQ-MVP1'> comparison : SimRequirement {
        doc /* The MVP shall compare the same VIPER-class rover trip with and without a road,
             * on real lunar south polar terrain, and report the difference in time, energy,
             * risk and reliability. */
    }

    requirement <'REQ-MVP2'> reportEitherWay : SimRequirement {
        doc /* Results shall be reported whichever way they turn out, including if the road
             * makes little difference. */
    }

    requirement <'REQ-MVP3'> caveat : SimRequirement {
        doc /* Every shared result shall state that it comes from a simplified model using
             * assumed rover loads, slip and road performance. VIPER is driven by a human team,
             * while this demo plans automatically. */
    }

    // ---------------------------------------------------------------------
    // Model 1: Mobility
    // ---------------------------------------------------------------------

    requirement <'REQ-M1'> roadSpeedAndSlip : SimRequirement {
        doc /* Speed made good and slip shall use the no-road values off-road, and the active
             * preset on road cells. */
    }

    requirement <'REQ-M2'> slopeLimit : SimRequirement {
        doc /* The planner shall reject cells above the 15 degree slope limit. */
        require constraint { sim.planner.maxPlannedSlope <= 15 ['°'] }
    }

    requirement <'REQ-M3'> headingForSun : SimRequirement {
        doc /* Body heading may differ from travel direction by up to plus or minus 50 degrees
             * (assumed) while driving.
             * In sunlight the rover shall choose the allowed heading with the most solar input;
             * in shadow it shall face its direction of travel. */
        require constraint { sim.rover.mobility.maxHeadingOffset <= 50 ['°'] }
    }

    // ---------------------------------------------------------------------
    // Model 2: Power
    // ---------------------------------------------------------------------

    requirement <'REQ-P1'> loadModel : SimRequirement {
        doc /* Load shall be the awake load, plus driving power multiplied by the moving fraction. */
    }

    requirement <'REQ-P2'> chargeBounds : SimRequirement {
        doc /* Charge shall stay between 0% and 100%, with excess solar power discarded. */
        require constraint {
            sim.rover.power.stateOfCharge >= 0.0 and sim.rover.power.stateOfCharge <= 1.0
        }
    }

    // ---------------------------------------------------------------------
    // Model 3: Environment and road maps
    // ---------------------------------------------------------------------

    requirement <'REQ-E1'> visibilityFromTerrain : SimRequirement {
        doc /* Sun and Earth visibility shall be calculated from terrain, not hand-painted. */
    }

    requirement <'REQ-E2'> visibilityHeights : SimRequirement {
        doc /* Earth visibility shall use 2 m height and sunlight 1 m height (both assumed). */
        require constraint {
            sim.environment.earthVisibilityHeight == 2 [m] and
            sim.environment.sunlightHeight == 1 [m]
        }
    }

    requirement <'REQ-E3'> roadOnlyChangesFlags : SimRequirement {
        doc /* Turning the road off shall change only the road flags. */
    }

    // ---------------------------------------------------------------------
    // Model 4: Command delay
    // ---------------------------------------------------------------------

    requirement <'REQ-C1'> delaySampling : SimRequirement {
        doc /* Round-trip delay shall be sampled from 6 to 10 s. Rover state and status messages
             * shall update every 60 s (assumed), with message delays handled as timed events. */
        require constraint {
            sim.commLink.minRoundTrip == 6 [s] and
            sim.commLink.maxRoundTrip == 10 [s]
        }
    }

    requirement <'REQ-C2'> commandHandshake : SimRequirement {
        doc /* The rover shall not start a new waypoint without a command, or while Earth is out
             * of view. A command with no acknowledgement within 10 s shall be resent, and a dropped
             * status message is replaced by the next one. Drops and resends shall be logged. */
        require constraint { sim.commLink.ackTimeout == 10 [s] }
    }

    // ---------------------------------------------------------------------
    // Model 5: Planner
    // ---------------------------------------------------------------------

    requirement <'REQ-R1'> timeAndEnergy : SimRequirement {
        doc /* The planner shall work out time and energy from the active speed, slip and load. */
    }

    requirement <'REQ-R2'> hardLimits : SimRequirement {
        doc /* Hard limits shall be slope, battery reserve, and no driving without Earth in view. */
    }

    requirement <'REQ-R3'> arrivalObjective : SimRequirement {
        doc /* The planner shall minimise arrival time, with an energy penalty of 0.001 h per Wh
             * drawn from the battery (assumed and configurable). */
    }

    requirement <'REQ-R4'> planTime : SimRequirement {
        doc /* The planner may use a coarser grid, but each plan shall finish within 5 min (assumed)
             * and the final route shall be rechecked at full resolution. */
        require constraint { sim.planner.maxPlanTime <= 5 [min] }
    }

    // ---------------------------------------------------------------------
    // Scenario, data and delivery (spec "shall" statements without IDs)
    // ---------------------------------------------------------------------

    requirement <'REQ-MVP4'> assumedValuesConfigurable : SimRequirement {
        doc /* Every assumed value shall be configurable. */
    }

    requirement <'REQ-MVP5'> fixedScenario : SimRequirement {
        doc /* Each comparison shall use one fixed start, goal, start time and terrain map. */
    }

    requirement <'REQ-MVP6'> widerTerrainForShadows : SimRequirement {
        doc /* The shadow calculation shall use a wider, lower-resolution terrain map around
             * the 2 km patch, because distant ridges cast shadows into the area. */
    }

    requirement <'REQ-MVP7'> siteAndPatchSelection : SimRequirement {
        doc /* The demo shall use Connecting Ridge. The 2 km patch and scenario date shall be
             * chosen from the data using the spec's selection rules, in order. */
    }

    requirement <'REQ-MVP8'> r2Framing : SimRequirement {
        doc /* Scenario R2 shall be described as selected to illustrate the problem, not as a
             * typical trip. */
    }

    requirement <'REQ-MVP9'> precomputedRunSet : SimRequirement {
        doc /* Every control combination shall be precomputed and stored, so the viewer changes
             * results without replanning: 3 no-road and 9 road runs per scenario, 24 in total. */
        require constraint { sim.display.storedRuns == 24 }
    }

    requirement <'REQ-MVP10'> displayIndependence : SimRequirement {
        doc /* The simulation core and stored run set shall stay independent of the final display. */
    }

    // Not modelled: the spec's handover duty that the simulation engineer shall propose
    // tools, schedule, budget and code ownership. It is a project duty, not a system requirement.

    // ---------------------------------------------------------------------
    // Specification group, used for the satisfy relation
    // ---------------------------------------------------------------------

    requirement simulationSpecification : SimRequirement {
        doc /* All MVP simulation requirements. */
        require comparison;
        require reportEitherWay;
        require caveat;
        require roadSpeedAndSlip;
        require slopeLimit;
        require headingForSun;
        require loadModel;
        require chargeBounds;
        require visibilityFromTerrain;
        require visibilityHeights;
        require roadOnlyChangesFlags;
        require delaySampling;
        require commandHandshake;
        require timeAndEnergy;
        require hardLimits;
        require arrivalObjective;
        require planTime;
        require assumedValuesConfigurable;
        require fixedScenario;
        require widerTerrainForShadows;
        require siteAndPatchSelection;
        require r2Framing;
        require precomputedRunSet;
        require displayIndependence;
    }
}


## 04_Behaviour

In [ ]:
package LRD_Behaviour {
    doc /*
     * Functional flow of one simulation run, and the rover's operating modes.
     * The failure conditions come from the spec: charge below reserve, a cell above
     * the slope limit, or no arrival by the end of the scenario.
     */

    public import LRD_Structure::*;

    // =====================================================================
    // One command cycle of a run: scenario in, run record out.
    // A full run repeats the plan, delay and drive steps every 5 min until the
    // rover arrives or fails; status returns to the planner through the delay (IF-05).
    // Not modelled as behaviour: command acknowledgement and the 10 s resend (REQ-C2,
    // verified by CK-6) and the minimum 5 min between commands.
    // =====================================================================

    action def RunScenario {
        in item settings : ScenarioSettings;
        out item record : RunRecord;

        action loadScenario {
            in item settings : ScenarioSettings = RunScenario::settings;
            out item config : ScenarioSettings;
        }

        flow from loadScenario.config to computeMaps.config;

        then action computeMaps {
            doc /* Sun and Earth visibility per cell from terrain horizons; road flags from the centreline. */
            in item config : ScenarioSettings;
            out item maps : EnvironmentMaps;
        }

        flow from computeMaps.maps to planRoute.maps;

        then action planRoute {
            doc /* Time-aware plan minimising arrival time plus an energy penalty, within hard limits. */
            in item maps : EnvironmentMaps;
            out item command : Command;
        }

        flow from planRoute.command to delayCommand.commandIn;

        then action delayCommand {
            doc /* Hold the command for half of a sampled 6 to 10 s round trip; optionally drop it. */
            in item commandIn : Command;
            out item commandOut : Command;
        }

        flow from delayCommand.commandOut to driveAndUpdatePower.command;

        then action driveAndUpdatePower {
            doc /* Move with slip, update battery as solar input minus load, and report status. */
            in item command : Command;
            out item status : RoverStatus;
        }

        flow from driveAndUpdatePower.status to logResults.status;

        then action logResults {
            doc /* Record route, state, solar power, load, events and the reported results. */
            in item status : RoverStatus;
            out item record : RunRecord = RunScenario::record;
        }
    }

    // A comparison is the same scenario run twice, changing only the road setting.
    action def RunComparison {
        in item settings : ScenarioSettings;
        action noRoadRun : RunScenario;
        then action roadRun : RunScenario;
    }

    // =====================================================================
    // Rover operating modes
    // =====================================================================

    attribute def CommandReceived;
    attribute def HoldCommanded;
    attribute def WaypointReached;
    attribute def GoalReached;
    attribute def EarthLost;
    attribute def EarthRegained;
    attribute def ChargeBelowReserve;
    attribute def SlopeLimitExceeded;
    attribute def ScenarioTimeout;

    state def RoverModes {
        first start then awaitingCommand;

        state awaitingCommand {
            doc /* No new waypoint starts without a command (REQ-C2). */
        }
        accept CommandReceived then driving;
        accept HoldCommanded then holding;
        accept EarthLost then waitingForEarth;
        accept ChargeBelowReserve then failed;
        accept ScenarioTimeout then failed;

        state driving;
        accept WaypointReached then awaitingCommand;
        accept GoalReached then arrived;
        accept ChargeBelowReserve then failed;
        accept SlopeLimitExceeded then failed;
        accept EarthLost then waitingForEarth;
        accept ScenarioTimeout then failed;

        state holding {
            doc /* Planner chose to wait, for example for a moving shadow to pass. */
        }
        accept CommandReceived then driving;
        accept EarthLost then waitingForEarth;
        accept ChargeBelowReserve then failed;
        accept ScenarioTimeout then failed;

        state waitingForEarth {
            doc /* No driving without Earth in view (REQ-R2). Entered from awaiting a command,
                 * holding or mid-drive; the rover stops until Earth is back in view. */
        }
        accept EarthRegained then awaitingCommand;
        accept ChargeBelowReserve then failed;
        accept ScenarioTimeout then failed;

        state arrived;
        state failed;
    }
}


## 05_RoadPresets_Variants

In [ ]:
package LRD_Variants {
    doc /*
     * Variability: the road presets and awake-load presets the viewer can choose.
     * All road values are assumed placeholders until test data or a source exists.
     * Trip times assume a straight 1.5 km route with no detours or waiting.
     * The stored run set is 3 no-road plus 9 road runs per scenario, 24 in total.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;
    public import LRD_Structure::*;

    part def RoadPreset {
        attribute speedMadeGoodFlat : SpeedValue;
        attribute slipMultiplier : Real;
        attribute rollingResistance : Real;
        attribute maxWaypointSpacing : LengthValue;
        attribute tripTimeStraight1500m : DurationValue;
    }

    // No-road speed made good and waypoint spacing are sourced ('='). Every other
    // road value is an assumed placeholder ('default'), and trip times are derived
    // from the assumed speeds.

    part noRoad : RoadPreset {
        attribute :>> speedMadeGoodFlat = 0.8 [cm/s];
        attribute :>> slipMultiplier default 1.0;
        attribute :>> rollingResistance default 0.2;
        attribute :>> maxWaypointSpacing = 8 [m];
        attribute :>> tripTimeStraight1500m default 52 [h];   // derived
    }

    part halfBenefit : RoadPreset {
        attribute :>> speedMadeGoodFlat default 1.2 [cm/s];
        attribute :>> slipMultiplier default 0.75;
        attribute :>> rollingResistance default 0.15;
        attribute :>> maxWaypointSpacing default 8 [m];
        attribute :>> tripTimeStraight1500m default 35 [h];   // derived
    }

    part expectedBenefit : RoadPreset {
        attribute :>> speedMadeGoodFlat default 1.6 [cm/s];
        attribute :>> slipMultiplier default 0.5;
        attribute :>> rollingResistance default 0.1;
        attribute :>> maxWaypointSpacing default 8 [m];
        attribute :>> tripTimeStraight1500m default 26 [h];   // derived
    }

    part highBenefit : RoadPreset {
        attribute :>> speedMadeGoodFlat default 3.2 [cm/s];
        attribute :>> slipMultiplier default 0.25;
        attribute :>> rollingResistance default 0.05;
        attribute :>> maxWaypointSpacing default 16 [m];
        attribute :>> tripTimeStraight1500m default 13 [h];   // derived
    }

    variation part def RoadPresetChoices :> RoadPreset {
        variant noRoad;
        variant halfBenefit;
        variant expectedBenefit;
        variant highBenefit;
    }

    variation attribute def AwakeLoadChoices :> PowerValue {
        variant attribute lowLoad = 150 [W];
        variant attribute nominalLoad = 250 [W];
        variant attribute highLoad = 350 [W];
    }

    enum def ScenarioKind {
        doc /* R1 baseline with no moving shadow on the main route; R2 with a real moving shadow. */
        R1_Baseline;
        R2_MovingShadow;
    }

    // The family of runs the viewer can select from
    abstract part demoRunFamily : LunarRoadDemoSystem {
        part roadPreset : RoadPresetChoices[1];
        attribute awakeLoadPreset : AwakeLoadChoices[1];
        attribute scenarioKind : ScenarioKind;
    }

    // Two example configurations from the run set
    part r1NoRoadNominal :> demoRunFamily {
        part redefines roadPreset = roadPreset::noRoad;
        attribute redefines awakeLoadPreset = awakeLoadPreset::nominalLoad;
        attribute redefines scenarioKind = ScenarioKind::R1_Baseline;
    }

    part r2ExpectedRoadNominal :> demoRunFamily {
        part redefines roadPreset = roadPreset::expectedBenefit;
        attribute redefines awakeLoadPreset = awakeLoadPreset::nominalLoad;
        attribute redefines scenarioKind = ScenarioKind::R2_MovingShadow;
    }
}


## 06_Hardware_1to10

In [ ]:
package LRD_Hardware {
    doc /*
     * Physical architecture of the 1:10 VIPER-look tabletop rover, and its mapping
     * to VIPER's published electrical blocks.
     * Sources: "Lunar Road Demo: 1:10 VIPER-Look Rover Hardware Design" and
     * "VIPER Electrical Architecture Reconstruction".
     * The model borrows VIPER block names only; it reproduces none of VIPER's design.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;
    public import LRD_Structure::*;

    // =====================================================================
    // Ports for the tabletop electrical design (connections E1 to E6)
    // =====================================================================

    port def PowerRail {
        doc /* 4.8 V battery bus, up to about 5.8 V off the charger. */
        out attribute voltage : ElectricPotentialDifferenceValue;
    }
    port def PwmPair      { doc /* 2 PWM lines per motor, plus shared enable (SLP). */ }
    port def MotorPower   { doc /* Motor power, from a driver channel to its gearmotor. */ }
    port def EncoderLink  { doc /* Encoder A and B back to the Pico (GP10 to GP17), powered from its 3V3.
                                 * Shares the wheel's 6-pin JST-SH cable with motor power. */ }
    port def BatterySense { doc /* Battery voltage through a 2:1 divider to GP26 (ADC0). */ }
    port def I2CBus       { doc /* I2C plus 3V3. */ }
    port def LedDrive     { doc /* One GPIO to a transistor; LEDs run from the battery rail. */ }
    item def TabletopStatus {
        doc /* The MVP's IF-05 fields, with battery voltage replacing state of charge. */
        attribute time : DurationValue;
        attribute x : LengthValue;
        attribute y : LengthValue;
        attribute heading : AngularMeasureValue;
        attribute mode : String;
        attribute commandStatus : String;
        attribute batteryVoltage : ElectricPotentialDifferenceValue;
    }

    port def WifiLink {
        doc /* Wi-Fi to a laptop; commands reuse the MVP's IF-04 fields, status uses TabletopStatus. */
        in item command : Command;
        out item status : TabletopStatus;
    }

    // =====================================================================
    // Parts
    // =====================================================================

    part def BatteryBox {
        doc /* 4 AA 2,500 mAh NiMH cells in a switched box. */
        attribute nominalVoltage : ElectricPotentialDifferenceValue = 4.8 [V];
        attribute energy : EnergyValue = 12 [W*h];
        port powerOut : PowerRail;
        port sense : BatterySense;
    }

    part def PicoController {
        doc /* Raspberry Pi Pico 2 W: command, data and control on one board. */
        attribute speedLoopRate : Real default 50.0;   // Hz, assumed
        attribute commsLossTimeout : DurationValue = 1 [s];
        port powerIn : ~PowerRail;
        port pwmDriver1 : PwmPair;
        port pwmDriver2 : PwmPair;
        port imuBus : I2CBus;
        port ledOut : LedDrive;
        port wifi : WifiLink;
        port batterySense : ~BatterySense;
        port encoderFL : ~EncoderLink;
        port encoderFR : ~EncoderLink;
        port encoderRL : ~EncoderLink;
        port encoderRR : ~EncoderLink;
    }

    part def MotorDriver {
        doc /* Adafruit DRV8833 breakout, 1.2 A continuous per channel, 2 motors each. */
        attribute continuousCurrentPerChannel : ElectricCurrentValue = 1.2 [A];
        port powerIn : ~PowerRail;
        port pwmIn : ~PwmPair;
        port motorA : MotorPower;
        port motorB : MotorPower;
    }

    part def WheelModule {
        doc /* 1006:1 micro metal gearmotor with Hall encoder and a printed grousered wheel. */
        attribute wheelDiameter : LengthValue = 50 [mm];
        attribute wheelWidth : LengthValue = 20 [mm];
        attribute stallTorqueDerated : TorqueValue = 0.23 [N*m];   // derived from 2.3 kg.cm at 4.8 V
        port motorIn : ~MotorPower;
        port encoder : EncoderLink;
    }

    part def Imu {
        doc /* 6-axis IMU breakout; heading by gyro only, so it drifts. */
        port bus : ~I2CBus;
    }

    part def MastLeds {
        doc /* Two white LEDs in the mast; camera holes are dummies. */
        port drive : ~LedDrive;
    }

    part def ChassisTub    { doc /* Load-carrying printed tub, 150 x 118 x 30 mm, 17 mm ground clearance. */ }
    part def BodyShell     { doc /* Cosmetic open-bottom shell with top hatch. */ }
    part def DummyPanel    { doc /* Printed solar panel with a cell grid; no power generation. */ }
    part def Mast          { doc /* T-shaped mast, top at 250 mm. */ }

    part def TabletopRover {
        doc /* 1:10 four-wheel skid-steer model, about 0.6 kg, top speed near 3 cm/s. */

        // Measured or calculated properties checked by the CK-H checks
        attribute footprint : LengthValue = 170 [mm];   // design value
        attribute mass : MassValue default 0.6 [kg];
        attribute topSpeed : SpeedValue = 3.0 [cm/s];
        attribute scaleSpeed : SpeedValue = 2 [cm/s];
        attribute drivingPower : PowerValue default 1.48 [W];   // derived from assumed Pico and LED/IMU loads
        attribute partsCostGBP : Real default 173.70;
        attribute measuredFootprint : LengthValue;
        attribute measuredWheelDiameter : LengthValue;
        attribute measuredWheelWidth : LengthValue;
        attribute measuredLevelSpeed : SpeedValue;
        attribute continuousRunTime : DurationValue;
        attribute commsLossStopTime : DurationValue;
        attribute maxClimbableSlope : AngularMeasureValue;

        // Structure
        part tub : ChassisTub;
        part shell : BodyShell;
        part panels : DummyPanel[3];
        part mast : Mast;

        // Electrical
        part battery : BatteryBox;
        part controller : PicoController;
        part driver1 : MotorDriver;
        part driver2 : MotorDriver;
        part frontLeft : WheelModule;
        part frontRight : WheelModule;
        part rearLeft : WheelModule;
        part rearRight : WheelModule;
        part imu : Imu;
        part leds : MastLeds;

        port wifi : WifiLink;
        bind wifi = controller.wifi;

        // E1: battery to both drivers directly, to the Pico's VSYS through a silicon diode,
        // and to the battery sense pin through a 2:1 divider
        connection 'E1 battery to driver 1' connect battery.powerOut to driver1.powerIn;
        connection 'E1 battery to driver 2' connect battery.powerOut to driver2.powerIn;
        connection 'E1 battery to Pico VSYS' connect battery.powerOut to controller.powerIn;
        connection 'E1 battery sense' connect battery.sense to controller.batterySense;

        // E2: Pico to drivers, 8 PWM lines and 1 shared enable
        connection 'E2 Pico to driver 1' connect controller.pwmDriver1 to driver1.pwmIn;
        connection 'E2 Pico to driver 2' connect controller.pwmDriver2 to driver2.pwmIn;

        // E3: control board to each wheel over one 6-pin JST-SH cable, carrying motor
        // power from a driver and encoder signals back to the Pico
        connection 'E3 front left power'  connect driver1.motorA to frontLeft.motorIn;
        connection 'E3 front right power' connect driver1.motorB to frontRight.motorIn;
        connection 'E3 rear left power'   connect driver2.motorA to rearLeft.motorIn;
        connection 'E3 rear right power'  connect driver2.motorB to rearRight.motorIn;
        connection 'E3 front left encoder'  connect frontLeft.encoder to controller.encoderFL;
        connection 'E3 front right encoder' connect frontRight.encoder to controller.encoderFR;
        connection 'E3 rear left encoder'   connect rearLeft.encoder to controller.encoderRL;
        connection 'E3 rear right encoder'  connect rearRight.encoder to controller.encoderRR;

        // E4 and E5
        connection 'E4 Pico to IMU'  connect controller.imuBus to imu.bus;
        connection 'E5 Pico to LEDs' connect controller.ledOut to leds.drive;
    }

    part def RoutePlayerLaptop {
        doc /* Reads a stored route (IF-06 format) and sends waypoints; can hold each message for
             * half of a sampled 6 to 10 s round trip. */
        port wifi : ~WifiLink;
    }

    part def OverheadCamera {
        doc /* USB webcam and printed hatch marker for position and heading correction (decision D3). */
    }

    part tabletopDemo {
        part rover : TabletopRover;
        part laptop : RoutePlayerLaptop;
        part camera : OverheadCamera;
        // E6: Pico to laptop over Wi-Fi
        connection 'E6 Wi-Fi link' connect rover.wifi to laptop.wifi;
    }

    // =====================================================================
    // VIPER reference blocks (public sources only; not simulated, not reproduced)
    // =====================================================================

    package ViperReference {
        part def SolarArray       { attribute powerPerPanel : PowerValue = 320 [W]; }
        part def ViperBattery     { doc /* One of two internal units; core components shared with Orion. */ }
        part def IntegratedAvionicsUnit { doc /* Command and data handling with electric power functions (Moog). */ }
        part def SEPIA            { doc /* With the IAU, manages solar array, battery charging and power distribution. */ }
        part def MotorControlBox  { doc /* Controls 6 BLDC actuators; 20 A peak per axis for 10 s; in the warm box. */ }
        part def ViperWheelModule { doc /* Drive, steering and suspension actuators. */ }
        part def ViperImu;
        part def StarTracker      { doc /* Rocket Lab ST16RT2; 28 V dual RS-485. */ }
        part def CameraSystem     { doc /* Eight cameras with a Camera Control Unit. */ }
        part def LightSystem      { doc /* Eight LED-array lights with a Light Controller. */ }
        part def XBandLink        { doc /* X-band direct to Earth, gimballed high-gain and fixed low-gain antennas. */ }

        part def ViperRover {
            attribute batteryCapacityStartOfLife : EnergyValue = 5420 [W*h];   // total of both units, 0 C
            part solarArrays : SolarArray[3];
            part batteries : ViperBattery[2];
            part iau : IntegratedAvionicsUnit;
            part sepia : SEPIA;
            part motorControlBoxes : MotorControlBox[1..*];   // count not published; two is an inference
            part wheelModules : ViperWheelModule[4];
            part imu : ViperImu;
            part starTracker : StarTracker;
            part cameras : CameraSystem;
            part lights : LightSystem;
            part radio : XBandLink;
        }
    }

    // =====================================================================
    // Mapping: which tabletop element stands in for which VIPER block
    // =====================================================================

    part viperReference : ViperReference::ViperRover;
    part tabletopRoverDesign : TabletopRover;

    allocation def StandsInFor {
        doc /* The tabletop element plays the role of the VIPER block in the demo only. */
        end viperBlock : Parts::Part;
        end modelElement : Parts::Part;
    }

    // Mapping follows the hardware spec's VIPER-to-tabletop table.
    allocation batteryMap : StandsInFor allocate viperReference.batteries to tabletopRoverDesign.battery;
    allocation avionicsMap : StandsInFor allocate viperReference.iau to tabletopRoverDesign.controller;
    allocation powerMgmtMap : StandsInFor allocate viperReference.sepia to tabletopRoverDesign.controller;
    allocation powerSwitchMap : StandsInFor allocate viperReference.sepia to tabletopRoverDesign.battery;
    // The motor control boxes, whatever their number, stand in as a set for both drivers.
    allocation motorCtrlMap1 : StandsInFor allocate viperReference.motorControlBoxes to tabletopRoverDesign.driver1;
    allocation motorCtrlMap2 : StandsInFor allocate viperReference.motorControlBoxes to tabletopRoverDesign.driver2;
    // VIPER's four wheel modules stand in as a set for the four tabletop wheels.
    allocation wheelMapFL : StandsInFor allocate viperReference.wheelModules to tabletopRoverDesign.frontLeft;
    allocation wheelMapFR : StandsInFor allocate viperReference.wheelModules to tabletopRoverDesign.frontRight;
    allocation wheelMapRL : StandsInFor allocate viperReference.wheelModules to tabletopRoverDesign.rearLeft;
    allocation wheelMapRR : StandsInFor allocate viperReference.wheelModules to tabletopRoverDesign.rearRight;
    allocation imuMap : StandsInFor allocate viperReference.imu to tabletopRoverDesign.imu;
    allocation starTrackerMap : StandsInFor allocate viperReference.starTracker to tabletopRoverDesign.imu;
    allocation camerasMap : StandsInFor allocate viperReference.cameras to tabletopRoverDesign.mast;
    allocation lightsMap : StandsInFor allocate viperReference.lights to tabletopRoverDesign.leds;
    allocation radioMap : StandsInFor allocate viperReference.radio to tabletopRoverDesign.controller;
    allocation panelsMap : StandsInFor allocate viperReference.solarArrays to tabletopRoverDesign.panels;

    use case def 'Replay Route On Tabletop' {
        subject rover : TabletopRover;
        actor laptop : RoutePlayerLaptop;
        objective {
            doc /* Drive a stored road or no-road route on a printed terrain board at 1:1,000 route scale,
                 * as a physical prop for the simulation results. */
        }
    }
}


## 07_Requirements_Hardware

In [ ]:
package LRD_Requirements_Hardware {
    doc /*
     * The ten requirements that define a finished 1:10 rover build (REQ-H1 to REQ-H10),
     * plus the spec's required caveat, given the ID REQ-H11 here.
     */

    private import ScalarValues::*;
    private import ISQ::*;
    private import SI::*;
    public import LRD_Hardware::*;

    requirement def HardwareRequirement {
        subject rover : TabletopRover;
    }

    requirement <'REQ-H1'> footprintAndWheels : HardwareRequirement {
        doc /* The rover shall fit a 170 mm x 170 mm footprint with 50 mm x 20 mm wheels, within 2 mm. */
        require constraint {
            rover.measuredFootprint >= 168 [mm] and rover.measuredFootprint <= 172 [mm] and
            rover.measuredWheelDiameter >= 48 [mm] and rover.measuredWheelDiameter <= 52 [mm] and
            rover.measuredWheelWidth >= 18 [mm] and rover.measuredWheelWidth <= 22 [mm]
        }
    }

    requirement <'REQ-H2'> viperFeatures : HardwareRequirement {
        doc /* The rover shall carry VIPER's recognisable features: a box body, a wheel at each corner,
             * panels on the rear, left and right, and a front mast. */
    }

    requirement <'REQ-H3'> printable : HardwareRequirement {
        doc /* All structural parts shall print in PLA on a 180 mm x 180 mm bed without supports,
             * except where the CAD section says otherwise. */
    }

    requirement <'REQ-H4'> levelSpeed : HardwareRequirement {
        doc /* The rover shall hold a commanded speed of 2 cm/s on level ground, within 10%. */
        require constraint {
            rover.measuredLevelSpeed >= 1.8 [cm/s] and rover.measuredLevelSpeed <= 2.2 [cm/s]
        }
    }

    requirement <'REQ-H5'> slopeAndTurn : HardwareRequirement {
        doc /* The rover shall climb a 15 degree slope and turn on the spot on the chosen demo surface. */
        require constraint { rover.maxClimbableSlope >= 15 ['°'] }
    }

    requirement <'REQ-H6'> endurance : HardwareRequirement {
        doc /* The rover shall run for at least 2 hours of continuous driving on one set of cells. */
        require constraint { rover.continuousRunTime >= 2 [h] }
    }

    requirement <'REQ-H7'> wifiCommanding : HardwareRequirement {
        doc /* The rover shall accept waypoint commands over Wi-Fi and report position, heading and
             * battery voltage. */
    }

    requirement <'REQ-H8'> safeStop : HardwareRequirement {
        doc /* The rover shall stop within 1 s if commands stop arriving. */
        require constraint { rover.commsLossStopTime <= 1 [s] }
    }

    requirement <'REQ-H9'> cost : HardwareRequirement {
        doc /* Total parts cost shall stay under GBP 300, including filament and delivery. */
        require constraint { rover.partsCostGBP < 300.0 }
    }

    requirement <'REQ-H10'> massLimit : HardwareRequirement {
        doc /* Mass shall stay under 0.8 kg, so the torque margins in the spec hold. */
        require constraint { rover.mass < 0.8 [kg] }
    }

    requirement <'REQ-H11'> scaleModelCaveat : HardwareRequirement {
        doc /* Anything shown with this rover shall state that it is a scale model driving a
             * replayed route, and that its speed, slip and energy use say nothing about VIPER
             * or a lunar road. */
    }

    requirement hardwareSpecification : HardwareRequirement {
        doc /* All 1:10 rover build requirements. */
        require footprintAndWheels;
        require viperFeatures;
        require printable;
        require levelSpeed;
        require slopeAndTurn;
        require endurance;
        require wifiCommanding;
        require safeStop;
        require cost;
        require massLimit;
        require scaleModelCaveat;
    }
}


## 08_Verification_Traceability

In [ ]:
package LRD_Verification {
    doc /*
     * Verification cases (CK-1 to CK-5 for the simulation and CK-H1 to CK-H8 for the rover,
     * from the specs, plus the proposed CK-6, CK-7 and CK-H9),
     * satisfy relations, and allocation of functions to logical elements.
     * None of the hardware checks has been run yet, because nothing is built.
     */

    private import VerificationCases::*;
    private import VerificationMethodKind::*;
    public import LRD_Requirements_Sim::*;
    public import LRD_Requirements_Hardware::*;
    public import LRD_Behaviour::*;

    // =====================================================================
    // Design configurations that the requirements are placed on
    // =====================================================================

    part mvpDesign : LunarRoadDemoSystem {
        part :>> rover {
            exhibit state modes : RoverModes;
        }
    }

    part roverBuild : TabletopRover;

    satisfy simulationSpecification by mvpDesign;
    satisfy hardwareSpecification by roverBuild;

    // =====================================================================
    // Allocation: which logical element performs each step of a run
    // =====================================================================

    action runScenario : RunScenario;

    allocate runScenario.computeMaps to mvpDesign.environment;
    allocate runScenario.planRoute to mvpDesign.planner;
    allocate runScenario.delayCommand to mvpDesign.commLink;
    allocate runScenario.driveAndUpdatePower to mvpDesign.rover;
    allocate runScenario.logResults to mvpDesign.display;

    // =====================================================================
    // Simulation checks (MVP spec, "MVP checks"). The spec does not say which
    // requirements each check covers, so these mappings are this model's judgement:
    // each check verifies only what its pass condition actually tests.
    // =====================================================================

    verification def 'CK-1 FairAndRepeatable' {
        doc /* Pass: both runs differ only by the road setting, and the same seed gives the same result. */
        @VerificationMethod { kind = test; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify roadOnlyChangesFlags;
            verify fixedScenario;
        }
    }

    verification def 'CK-2 LimitsRespected' {
        doc /* Pass: no route enters a cell above 15 degrees or drives without Earth in view.
             * Covers the slope and Earth parts of REQ-R2 and REQ-C2 only; the reserve limit,
             * resends and logging are covered by CK-6. */
        @VerificationMethod { kind = (test, analyze); }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify slopeLimit;
            verify hardLimits;
            verify commandHandshake;
        }
    }

    verification def 'CK-3 PowerModelCorrect' {
        doc /* Pass: Sun on a rear corner gives about 450 W, and a 600 W test load drains a full
             * battery in about 9 h. The second check tests code only. */
        @VerificationMethod { kind = (analyze, test); }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify loadModel;
            verify chargeBounds;
        }
    }

    verification def 'CK-4 ShadowsBelievable' {
        doc /* Pass: shadows come from terrain, and most shadow-edge speeds fall within the
             * published 0.1 to 1.8 cm/s. */
        @VerificationMethod { kind = analyze; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify visibilityFromTerrain;
            verify widerTerrainForShadows;
        }
    }

    verification def 'CK-5 DemoWorksEndToEnd' {
        doc /* Pass: a viewer can change every control and see route, charge, shadow and results
             * update from the stored run set. */
        @VerificationMethod { kind = demo; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify precomputedRunSet;
        }
    }

    // Checks CK-6 and CK-7 are proposed by this model, not in the original spec.
    // They cover the requirements that CK-1 to CK-5 do not test.

    verification def 'CK-6 PROPOSED PlannerPowerAndDelayTests' {
        doc /* Proposed: unit tests that the planner uses the active preset on road cells, picks
             * the sunniest allowed heading, respects the battery reserve, applies the energy
             * penalty and finishes each plan within 5 min; that delay is sampled from 6 to 10 s;
             * that unacknowledged commands are resent after 10 s and drops are logged; and that
             * visibility uses the 2 m and 1 m heights. */
        @VerificationMethod { kind = test; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify roadSpeedAndSlip;
            verify headingForSun;
            verify hardLimits;
            verify timeAndEnergy;
            verify arrivalObjective;
            verify planTime;
            verify delaySampling;
            verify commandHandshake;
            verify visibilityHeights;
        }
    }

    verification def 'CK-7 PROPOSED ResultsAndDeliveryReview' {
        doc /* Proposed: review that results report time, energy, risk and reliability either way,
             * carry the caveat, describe R2 as selected, use Connecting Ridge with the selection
             * rules, that the run set works without the final display, and that the scenario
             * configuration exposes every assumed value. */
        @VerificationMethod { kind = inspect; }
        subject sim : LunarRoadDemoSystem;
        objective {
            verify comparison;
            verify reportEitherWay;
            verify caveat;
            verify r2Framing;
            verify siteAndPatchSelection;
            verify displayIndependence;
            verify assumedValuesConfigurable;
        }
    }

    // =====================================================================
    // Hardware checks (1:10 rover spec, "Checks")
    // =====================================================================

    verification def 'CK-H1 MeasureFinishedRover' {
        doc /* Pass: 170 x 170 mm footprint and 50 x 20 mm wheels within 2 mm; all features present. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective { verify footprintAndWheels; verify viperFeatures; }
    }

    verification def 'CK-H2 PrintEveryPart' {
        doc /* Pass: each part prints on a 180 mm bed without supports. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective { verify printable; }
    }

    verification def 'CK-H3 LevelSpeedRun' {
        doc /* Drive 1 m at a commanded 2 cm/s on level ground. Pass: takes 46 to 55 s. */
        @VerificationMethod { kind = test; }
        subject rover : TabletopRover;
        objective { verify levelSpeed; }
    }

    verification def 'CK-H4 RampAndTurn' {
        doc /* Drive up a 15 degree ramp of the demo surface, then turn 360 degrees on the level.
             * Pass: climbs without stalling and completes the turn. */
        @VerificationMethod { kind = test; }
        subject rover : TabletopRover;
        objective { verify slopeAndTurn; }
    }

    verification def 'CK-H5 EnduranceRun' {
        doc /* Drive continuously from freshly charged cells. Pass: still driving after 2 h. */
        @VerificationMethod { kind = test; }
        subject rover : TabletopRover;
        objective { verify endurance; }
    }

    verification def 'CK-H6 WaypointsAndCommsLoss' {
        doc /* Send five waypoints over Wi-Fi, then stop sending. Pass: status messages return and
             * the rover stops within 1 s of the last message. */
        @VerificationMethod { kind = test; }
        subject rover : TabletopRover;
        objective { verify wifiCommanding; verify safeStop; }
    }

    verification def 'CK-H7 TotalReceipts' {
        doc /* Pass: under GBP 300. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective { verify cost; }
    }

    verification def 'CK-H8 WeighRover' {
        doc /* Pass: under 0.8 kg. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective { verify massLimit; }
    }

    verification def 'CK-H9 PROPOSED CaveatCheck' {
        doc /* Proposed: check that anything shown with the rover carries the scale-model caveat. */
        @VerificationMethod { kind = inspect; }
        subject rover : TabletopRover;
        objective { verify scaleModelCaveat; }
    }
}


## Diagrams

### System context (IF-01, IF-02)

In [ ]:
%viz --view=INTERCONNECTION LRD_Structure::demoContext

### Logical architecture with IF-03 to IF-07

In [ ]:
%viz --view=INTERCONNECTION LRD_Structure::LunarRoadDemoSystem

### Part tree

In [ ]:
%viz --view=TREE LRD_Structure::LunarRoadDemoSystem

### One simulation run

In [ ]:
%viz --view=ACTION LRD_Behaviour::RunScenario

### Rover operating modes

In [ ]:
%viz --view=STATE LRD_Behaviour::RoverModes

### Simulation requirements

In [ ]:
%viz --view=TREE LRD_Requirements_Sim::simulationSpecification

### Road preset variants

In [ ]:
%viz --view=TREE LRD_Variants::RoadPresetChoices

### 1:10 rover electrical design (E1 to E6)

In [ ]:
%viz --view=INTERCONNECTION LRD_Hardware::TabletopRover

### VIPER block to tabletop mapping

In [ ]:
%viz --view=TREE LRD_Hardware::viperReference LRD_Hardware::tabletopRoverDesign

### Hardware requirements

In [ ]:
%viz --view=TREE LRD_Requirements_Hardware::hardwareSpecification